# Attention Benchmark for AGI Evaluation

This notebook evaluates a language model's attention capabilities across three cognitive tasks drawn from human neuropsychology:

- **Selective Attention** — The model is presented with a passage containing embedded distractors and must extract only goal-relevant information, filtering out irrelevant content.
- **Sustained Attention** — The model must track and aggregate specific targets accurately across a long sequential context without skipping or approximating.
- **Divided Attention** — The model must simultaneously monitor and reason across two independent information streams without conflating them.

Together these tasks probe whether a model demonstrates attentional control — the ability to filter noise, sustain focus over length, and split cognitive resources across streams — core faculties in AGI evaluation.

> **Environment note:** This notebook is designed to run inside a Kaggle notebook environment where `kaggle_benchmarks` is pre-installed and `kbench.llm` is auto-configured via injected credentials. It will not make live API calls outside that environment.

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd

In [ ]:
SELECTIVE_TASK_DATA = pd.DataFrame([
    {
        "context": (
            "The annual report stated that revenue grew by 12% in Q3. "
            "The marketing team launched three new campaigns in August. "
            "Operating costs fell by 4% due to supply chain improvements. "
            "The CEO mentioned plans to expand into Asian markets next year. "
            "Employee headcount increased by 200 in the past quarter. "
            "Net profit margin improved from 8.1% to 9.6% year-over-year."
        ),
        "question": "What financial performance metrics are reported?",
        "distractors": ["marketing campaigns", "Asian market expansion", "headcount"],
        "criteria": [
            "Response includes the 12% revenue growth in Q3.",
            "Response includes the 4% reduction in operating costs.",
            "Response includes the improvement in net profit margin from 8.1% to 9.6%.",
            "Response does not focus on marketing campaigns or headcount as financial metrics.",
        ],
    },
    {
        "context": (
            "The patient presented with a fever of 101.3F and mild cough. "
            "She mentioned her dog recently had surgery for a broken leg. "
            "Blood pressure was recorded at 118/76 mmHg. "
            "Her teenage son started college last month. "
            "Oxygen saturation measured at 97%. "
            "The patient reported occasional headaches over the past week. "
            "Her neighbor recommended a new restaurant downtown."
        ),
        "question": "What are the patient's clinical observations?",
        "distractors": ["dog surgery", "son starting college", "restaurant recommendation"],
        "criteria": [
            "Response includes the fever of 101.3F.",
            "Response includes the blood pressure reading of 118/76 mmHg.",
            "Response includes oxygen saturation of 97%.",
            "Response includes the mild cough and occasional headaches.",
            "Response excludes irrelevant personal details about the dog, son, or restaurant.",
        ],
    },
    {
        "context": (
            "The experiment used a sample size of 342 participants aged 18-65. "
            "The lead researcher enjoys hiking on weekends. "
            "Treatment group showed a 23% reduction in symptoms after 8 weeks. "
            "Data collection took place across four university hospitals. "
            "The control group received a placebo with no significant changes observed. "
            "Funding was provided by a private health foundation. "
            "Three participants withdrew due to personal reasons unrelated to the study."
        ),
        "question": "What were the key experimental findings and methodology?",
        "distractors": ["researcher hobbies", "funding source", "withdrawal reasons"],
        "criteria": [
            "Response includes the sample size of 342 participants.",
            "Response includes the 23% symptom reduction in the treatment group.",
            "Response includes the placebo control group outcome.",
            "Response includes the 8-week treatment duration.",
            "Response does not dwell on the researcher's hobbies or unrelated withdrawal reasons.",
        ],
    },
])


@kbench.task(name="selective_attention")
def selective_attention(
    llm,
    context: str,
    question: str,
    distractors: list,
    criteria: list,
):
    """
    Evaluating selective attention: model must extract goal-relevant information
    from a passage containing embedded distractors.
    """
    prompt = (
        f"Read the following passage carefully and answer the question. "
        f"Focus only on information directly relevant to the question.\n\n"
        f"Passage:\n{context}\n\n"
        f"Question: {question}"
    )

    response = llm.prompt(prompt)

    kbench.assertions.assess_response_with_judge(
        criteria=criteria,
        response_text=response,
        judge_llm=llm,
    )


selective_runs = []
for i, row in SELECTIVE_TASK_DATA.iterrows():
    run = selective_attention.run(
        llm=kbench.llm,
        context=row["context"],
        question=row["question"],
        distractors=row["distractors"],
        criteria=row["criteria"],
    )
    selective_runs.append((i, run))
    passed = sum(ar.passed for ar in run.assertion_results)
    total = len(run.assertion_results)
    print(f"Row {i}: {run.status.name} — {passed}/{total} assertions passed")

In [ ]:
SUSTAINED_TASK_DATA = pd.DataFrame([
    {
        "context": (
            "Transaction 1: Alice sent $200 to Bob. "
            "Transaction 2: Carol bought groceries for $45. "
            "Transaction 3: Bob transferred $80 to Dave. "
            "Transaction 4: Eve paid $120 for utilities. "
            "Transaction 5: Alice received $300 from Frank. "
            "Transaction 6: Dave spent $60 at a restaurant. "
            "Transaction 7: Carol received $150 from Grace. "
            "Transaction 8: Frank paid $90 for insurance. "
            "Transaction 9: Bob bought electronics for $400. "
            "Transaction 10: Eve received $250 from Alice. "
            "Transaction 11: Grace paid $30 for a subscription. "
            "Transaction 12: Dave received $110 from Carol. "
            "Transaction 13: Alice paid $75 for phone bills. "
            "Transaction 14: Frank spent $200 on travel. "
            "Transaction 15: Bob received $95 from Eve."
        ),
        "question": "What is Alice's net balance across all transactions? Show your working.",
        "expected_answer": "-75",
        "criteria": [
            "Response correctly identifies all transactions involving Alice.",
            "Response accounts for Alice sending $200 (debit).",
            "Response accounts for Alice receiving $300 from Frank (credit).",
            "Response accounts for Alice paying $75 for phone bills (debit).",
            "Response accounts for Eve receiving $250 from Alice (debit).",
            "Final net balance calculated is negative $75.",
        ],
    },
    {
        "context": (
            "Day 1: Temperature recorded at 18C, humidity 65%. "
            "Day 2: Temperature recorded at 21C, humidity 70%. "
            "Day 3: Temperature recorded at 17C, humidity 80%. "
            "Day 4: Temperature recorded at 23C, humidity 55%. "
            "Day 5: Temperature recorded at 19C, humidity 60%. "
            "Day 6: Temperature recorded at 25C, humidity 45%. "
            "Day 7: Temperature recorded at 22C, humidity 50%. "
            "Day 8: Temperature recorded at 16C, humidity 85%. "
            "Day 9: Temperature recorded at 20C, humidity 75%. "
            "Day 10: Temperature recorded at 24C, humidity 40%. "
            "Day 11: Temperature recorded at 18C, humidity 68%. "
            "Day 12: Temperature recorded at 21C, humidity 72%. "
            "Day 13: Temperature recorded at 26C, humidity 38%. "
            "Day 14: Temperature recorded at 15C, humidity 90%."
        ),
        "question": "How many days had both temperature above 20C and humidity below 60%?",
        "expected_answer": "3",
        "criteria": [
            "Response correctly identifies Day 6 as meeting both conditions (25C, 45%).",
            "Response correctly identifies Day 10 as meeting both conditions (24C, 40%).",
            "Response correctly identifies Day 13 as meeting both conditions (26C, 38%).",
            "Response correctly excludes days meeting only one condition.",
            "Final count given is 3 days.",
        ],
    },
    {
        "context": (
            "Item A: Category=Electronics, Price=$299, Stock=14 units. "
            "Item B: Category=Clothing, Price=$49, Stock=200 units. "
            "Item C: Category=Electronics, Price=$899, Stock=6 units. "
            "Item D: Category=Food, Price=$12, Stock=500 units. "
            "Item E: Category=Clothing, Price=$89, Stock=75 units. "
            "Item F: Category=Electronics, Price=$199, Stock=30 units. "
            "Item G: Category=Food, Price=$8, Stock=1000 units. "
            "Item H: Category=Clothing, Price=$129, Stock=40 units. "
            "Item I: Category=Electronics, Price=$649, Stock=9 units. "
            "Item J: Category=Food, Price=$22, Stock=300 units."
        ),
        "question": "What is the total inventory value of all Electronics items?",
        "expected_answer": "$18212",
        "criteria": [
            "Response identifies all four Electronics items: A, C, F, and I.",
            "Response correctly calculates Item A value as $4186 (299 x 14).",
            "Response correctly calculates Item C value as $5394 (899 x 6).",
            "Response correctly calculates Item F value as $5970 (199 x 30).",
            "Response correctly calculates Item I value as $5841 (649 x 9).",
            "Total inventory value calculated is $21391.",
        ],
    },
])


@kbench.task(name="sustained_attention")
def sustained_attention(
    llm,
    context: str,
    question: str,
    expected_answer: str,
    criteria: list,
):
    """
    Evaluating sustained attention: model must track and aggregate specific
    targets accurately across a long sequential context.
    """
    prompt = (
        f"Read the following records carefully and answer the question. "
        f"Track every relevant entry — do not skip or approximate.\n\n"
        f"Records:\n{context}\n\n"
        f"Question: {question}"
    )

    response = llm.prompt(prompt)

    kbench.assertions.assert_contains_regex(
        pattern=expected_answer,
        text=response,
        expectation=f"Response should contain the correct answer: {expected_answer}",
    )

    kbench.assertions.assess_response_with_judge(
        criteria=criteria,
        response_text=response,
        judge_llm=llm,
    )


sustained_runs = []
for i, row in SUSTAINED_TASK_DATA.iterrows():
    run = sustained_attention.run(
        llm=kbench.llm,
        context=row["context"],
        question=row["question"],
        expected_answer=row["expected_answer"],
        criteria=row["criteria"],
    )
    sustained_runs.append((i, run))
    passed = sum(ar.passed for ar in run.assertion_results)
    total = len(run.assertion_results)
    print(f"Row {i}: {run.status.name} — {passed}/{total} assertions passed")

In [ ]:
DIVIDED_TASK_DATA = pd.DataFrame([
    {
        "stream_a": (
            "Server A logs: "
            "09:00 - Request received from user_42. "
            "09:02 - Query executed successfully. "
            "09:05 - Request received from user_17. "
            "09:07 - ERROR: Timeout on database connection. "
            "09:10 - Request received from user_99. "
            "09:12 - Query executed successfully. "
            "09:15 - ERROR: Memory limit exceeded. "
            "09:18 - Request received from user_42. "
            "09:20 - Query executed successfully."
        ),
        "stream_b": (
            "Server B logs: "
            "09:01 - Request received from user_55. "
            "09:03 - Query executed successfully. "
            "09:06 - Request received from user_23. "
            "09:08 - Query executed successfully. "
            "09:11 - ERROR: Disk read failure. "
            "09:13 - Request received from user_77. "
            "09:16 - Query executed successfully. "
            "09:19 - ERROR: Timeout on database connection. "
            "09:21 - Request received from user_55."
        ),
        "question": (
            "Monitoring both server logs simultaneously: "
            "How many errors occurred on each server, "
            "and which error type appeared on both servers?"
        ),
        "criteria": [
            "Response correctly identifies 2 errors on Server A.",
            "Response correctly identifies 2 errors on Server B.",
            "Response identifies 'Timeout on database connection' as appearing on both servers.",
            "Response does not confuse errors between the two servers.",
            "Response addresses both streams without omitting either.",
        ],
    },
    {
        "stream_a": (
            "Portfolio A trades: "
            "Trade 1: Bought 50 shares of AAPL at $178. "
            "Trade 2: Sold 30 shares of GOOG at $142. "
            "Trade 3: Bought 100 shares of MSFT at $415. "
            "Trade 4: Sold 50 shares of AAPL at $185. "
            "Trade 5: Bought 20 shares of AMZN at $192."
        ),
        "stream_b": (
            "Portfolio B trades: "
            "Trade 1: Bought 80 shares of TSLA at $245. "
            "Trade 2: Bought 40 shares of GOOG at $140. "
            "Trade 3: Sold 80 shares of TSLA at $261. "
            "Trade 4: Sold 40 shares of NVDA at $875. "
            "Trade 5: Bought 60 shares of MSFT at $418."
        ),
        "question": (
            "Tracking both portfolios simultaneously: "
            "Which stock appears in both portfolios, "
            "and which portfolio realized a profit on its sold positions?"
        ),
        "criteria": [
            "Response correctly identifies GOOG and MSFT as appearing in both portfolios.",
            "Response correctly identifies Portfolio A realized profit on AAPL (bought at $178, sold at $185).",
            "Response correctly identifies Portfolio B realized profit on TSLA (bought at $245, sold at $261).",
            "Response correctly identifies Portfolio B realized profit on NVDA sold position.",
            "Response does not conflate trades between the two portfolios.",
        ],
    },
    {
        "stream_a": (
            "Team A sprint updates: "
            "Monday: Completed user authentication module. "
            "Tuesday: Started API integration, blocked on missing credentials. "
            "Wednesday: Credentials received, API integration resumed. "
            "Thursday: API integration completed and tested. "
            "Friday: Began dashboard UI development."
        ),
        "stream_b": (
            "Team B sprint updates: "
            "Monday: Completed database schema design. "
            "Tuesday: Started data migration scripts. "
            "Wednesday: Data migration 60% complete, no blockers. "
            "Thursday: Data migration completed. "
            "Friday: Started integration testing with Team A's API."
        ),
        "question": (
            "Tracking both team streams simultaneously: "
            "Which team experienced a blocker this sprint, "
            "and on which day did the two teams' work become directly dependent on each other?"
        ),
        "criteria": [
            "Response correctly identifies Team A as experiencing a blocker on Tuesday.",
            "Response correctly identifies the blocker as missing API credentials.",
            "Response correctly identifies Friday as the day Team B began depending on Team A's API.",
            "Response does not attribute Team B's progress as blocked.",
            "Response addresses both team streams without omitting either.",
        ],
    },
])


@kbench.task(name="divided_attention")
def divided_attention(
    llm,
    stream_a: str,
    stream_b: str,
    question: str,
    criteria: list,
):
    """
    Evaluating divided attention: model must simultaneously monitor and reason
    across two independent information streams without conflating them.
    """
    prompt = (
        f"You are given two independent information streams. "
        f"Read both carefully and answer the question by tracking each stream separately.\n\n"
        f"Stream A:\n{stream_a}\n\n"
        f"Stream B:\n{stream_b}\n\n"
        f"Question: {question}"
    )

    response = llm.prompt(prompt)

    kbench.assertions.assess_response_with_judge(
        criteria=criteria,
        response_text=response,
        judge_llm=llm,
    )


divided_runs = []
for i, row in DIVIDED_TASK_DATA.iterrows():
    run = divided_attention.run(
        llm=kbench.llm,
        stream_a=row["stream_a"],
        stream_b=row["stream_b"],
        question=row["question"],
        criteria=row["criteria"],
    )
    divided_runs.append((i, run))
    passed = sum(ar.passed for ar in run.assertion_results)
    total = len(run.assertion_results)
    print(f"Row {i}: {run.status.name} — {passed}/{total} assertions passed")

In [ ]:
def run_to_record(task_name, row_index, run):
    passed = sum(ar.passed for ar in run.assertion_results)
    total = len(run.assertion_results)
    assertion_summary = "; ".join(
        f"{'PASS' if ar.passed else 'FAIL'}: {ar.expectation}"
        for ar in run.assertion_results
    )
    return {
        "task": task_name,
        "row": row_index,
        "status": run.status.name,
        "assertions_passed": passed,
        "assertions_total": total,
        "pass_rate": round(passed / total, 2) if total > 0 else None,
        "error_message": run.error_message,
        "assertion_details": assertion_summary,
    }


records = []
for i, run in selective_runs:
    records.append(run_to_record("selective_attention", i, run))
for i, run in sustained_runs:
    records.append(run_to_record("sustained_attention", i, run))
for i, run in divided_runs:
    records.append(run_to_record("divided_attention", i, run))

results_df = pd.DataFrame(records)
pd.set_option("display.max_colwidth", 100)
results_df

## Results Interpretation

The benchmark reveals distinct attentional profiles across the three task types:

- **Selective Attention** rows test whether the model ignores clearly embedded distractors. A model with poor selective attention will incorporate irrelevant content (marketing campaigns, personal anecdotes, funding sources) into answers that should focus strictly on target information. Failures appear as assertion misses on the distractor-exclusion criteria.

- **Sustained Attention** rows test numerical tracking over extended sequences. Failures typically appear as omitted entries, rounding errors, or early stopping before all records are processed — hallmarks of attention drift over length. The `assert_contains_regex` assertion on `expected_answer` provides a hard signal independent of judge scoring.

- **Divided Attention** rows test simultaneous stream monitoring. A model failing divided attention will conflate information between Stream A and Stream B — attributing errors to the wrong server, mixing portfolio trades, or merging team timelines. These failures are qualitatively different from selective failures: the model has the information but assigns it incorrectly.

A model with strong attentional control should pass the majority of assertions across all three task types. Systematic failures within a single task type indicate a specific attentional weakness rather than general capability degradation, and provide a targeted signal for model comparison and iteration.